In [0]:
# Imports

from pyspark.sql.functions import (
    col, round as spark_round,
    avg, count,
    current_timestamp, date_format
)
from delta.tables import DeltaTable

print("Imports successful")

In [0]:
# Read Bronze climate and dim_country
df_bronze_climate = spark.table("workspace.default.bronze_open_meteo_climate")
df_dim_country = spark.table("workspace.default.dim_country")

print(f"Bronze climate rows: {df_bronze_climate.count()}")
print(f"dim_country rows:    {df_dim_country.count()}")

print("\nBronze climate schema:")
df_bronze_climate.printSchema()


In [0]:
display(df_bronze_climate)

In [0]:
display(df_dim_country)

In [0]:
# Aggregate daily to monthly

df_monthly_climate = (
    df_bronze_climate
    .groupBy("country_code", "year", "month")
    .agg(
        spark_round(avg("precipitation_mm"), 2).alias("avg_precipitation_mm"),
        spark_round(avg("temperature_c_mean"), 2).alias("avg_temperature_c"),
        count("*").alias("days_in_month")
    )
)

print(f"Monthly aggregated rows: {df_monthly_climate.count()}")

display(
    df_monthly_climate
    .filter(col("country_code") == "KE")
    .orderBy("year", "month")
    .limit(12)
)

In [0]:
# Join against dim_country

df_silver_climate = (
    df_monthly_climate
    .join(
        df_dim_country.select("country_code", "country_iso3", "country_name", "region"),
        on="country_code",
        how="inner"
    )
    .select(
        "country_code",
        "country_iso3",
        "country_name",
        "region",
        "year",
        "month",
        "avg_precipitation_mm",
        "avg_temperature_c",
        "days_in_month"
    )
)

df_silver_climate = df_silver_climate.withColumn(
    "updated_at",
    date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)

print(f"Silver climate rows after join: {df_silver_climate.count()}")
print(f"\nSample — Kenya 2023:")
display(
    df_silver_climate
    .filter((col("country_code") == "KE") & (col("year") == 2023))
    .orderBy("month")
)

In [0]:
# First-time write (create the Silver table)

TABLE_NAME = "workspace.default.silver_fact_climate"

(df_silver_climate.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Silver climate table created: {TABLE_NAME}")
print(f"Row count: {spark.table(TABLE_NAME).count()}")

In [0]:
# MERGE INTO

TABLE_NAME = "workspace.default.silver_fact_climate"

delta_target = DeltaTable.forName(spark, TABLE_NAME)

(
    delta_target.alias("target")
    .merge(
        df_silver_climate.alias("source"),
        """target.country_code = source.country_code
           AND target.year = source.year
           AND target.month = source.month"""
    )
    .whenMatchedUpdate(set={
        "avg_precipitation_mm": "source.avg_precipitation_mm",
        "avg_temperature_c":    "source.avg_temperature_c",
        "days_in_month":        "source.days_in_month",
        "country_iso3":         "source.country_iso3",
        "country_name":         "source.country_name",
        "region":               "source.region",
        "updated_at":           "source.updated_at",
    })
    .whenNotMatchedInsert(values={
        "country_code":         "source.country_code",
        "country_iso3":         "source.country_iso3",
        "country_name":         "source.country_name",
        "region":               "source.region",
        "year":                 "source.year",
        "month":                "source.month",
        "avg_precipitation_mm": "source.avg_precipitation_mm",
        "avg_temperature_c":    "source.avg_temperature_c",
        "days_in_month":        "source.days_in_month",
        "updated_at":           "source.updated_at",
    })
    .execute()
)

print(f"Row count after MERGE: {spark.table(TABLE_NAME).count()}")

In [0]:
# Verify

from pyspark.sql.functions import min, max

df_check = spark.table("workspace.default.silver_fact_climate")

print(f"Total rows: {df_check.count()}")

print("\nRows per country (expected 168 each):")
display(
    df_check.groupBy("country_code", "country_name", "region")
            .count()
            .orderBy("region", "country_name")
)

In [0]:
# Climate ranges by country
display(
    df_check.groupBy("country_code", "country_name")
            .agg(
                spark_round(avg("avg_precipitation_mm"), 2).alias("avg_rain_mm"),
                spark_round(avg("avg_temperature_c"),    2).alias("avg_temp_c"),
            )
            .orderBy("avg_rain_mm", ascending=False)
)

In [0]:
# Kenya monthly rainfall pattern 2023

display(
    df_check.filter(
        (col("country_code") == "KE") & (col("year") == 2023)
    )
    .select("month", "avg_precipitation_mm", "avg_temperature_c")
    .orderBy("month")
)